# Give Me Some Credit — EDA và Random Forest Grid Search

**Vai trò:** EDA và baseline Random Forest tập trung theo Protocol v1.0 trong
`ke_hoach_tong_hop_federated_random_forest_azure.md`. Đây chưa phải notebook
tổng kết FL/Azure. Chỉ so sánh kết quả này với FL khi cùng source, split,
feature schema, imputation, tổng số cây và tham số RF.

**Mục tiêu:** dự đoán `SeriousDlqin2yrs = 1` (trễ hạn nghiêm trọng trong 2 năm),
quan sát chất lượng dữ liệu, tìm tham số Random Forest theo **Average Precision**,
chọn ngưỡng trên validation và đánh giá đúng một lần trên test giữ tỷ lệ nhãn gốc.

**Chạy trên Kaggle:** thêm bộ dữ liệu *Give Me Some Credit* vào notebook bằng **Add Input**,
bật CPU, rồi chọn **Run All**. Notebook tự tìm `cs-training.csv` trong `/kaggle/input`.
Nếu có nhiều tệp trùng tên, đặt `DATA_PATH` ở ô cấu hình bên dưới.

Không cần Internet hoặc thư viện ngoài môi trường Python thường có trên Kaggle.
Thời gian Grid Search phụ thuộc CPU; mặc định là 8 cấu hình × 3 folds = 24 lần fit.

## 1. Thiết lập

In [ ]:
from pathlib import Path
import os
import hashlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

SEED = 42
TARGET = "SeriousDlqin2yrs"
DATA_PATH = None  # Ví dụ: "/kaggle/input/give-me-some-credit/cs-training.csv"
TEST_SIZE = 0.20
VALIDATION_FRACTION_OF_TRAIN = 0.25  # 90k fit / 30k validation / 30k test
MIN_RECALL_FOR_PRECISION = 0.30
CV_FOLDS = 3
GRID_N_JOBS = min(4, os.cpu_count() or 1)  # Giới hạn bộ nhớ khi chạy song song.

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13})
print(f"pandas={pd.__version__}, sklearn={__import__('sklearn').__version__}")

## 2. Đọc và kiểm tra dữ liệu

In [ ]:
if DATA_PATH is not None:
    csv_path = Path(DATA_PATH)
else:
    search_roots = [Path("/kaggle/input"), Path("data"), Path(".")]
    candidates = sorted({p.resolve() for root in search_roots if root.exists()
                         for p in root.rglob("cs-training.csv")})
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Cần đúng một cs-training.csv, tìm thấy {len(candidates)}: {candidates[:10]}. "
            "Hãy đặt DATA_PATH bằng đường dẫn CSV cụ thể."
        )
    csv_path = candidates[0]
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)
raw = pd.read_csv(csv_path)
raw = raw.drop(columns=[c for c in raw if str(c).startswith("Unnamed:")])
if raw.columns.duplicated().any() or TARGET not in raw:
    raise ValueError("Tên cột trùng hoặc thiếu cột nhãn SeriousDlqin2yrs")
if raw[TARGET].isna().any() or set(raw[TARGET].unique()) != {0, 1}:
    raise ValueError("Cột nhãn phải có đủ 0 và 1, không có giá trị thiếu")
for col in raw.columns:
    raw[col] = pd.to_numeric(raw[col], errors="raise")
features = [c for c in raw if c != TARGET]
if raw[features].isna().all().any():
    raise ValueError("Có đặc trưng trống hoàn toàn")
if np.isinf(raw[features].to_numpy(dtype=float)).any():
    raise ValueError("Dữ liệu có giá trị vô cực")

source_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
feature_schema_sha256 = hashlib.sha256("\n".join(features).encode("utf-8")).hexdigest()
print(f"source_sha256={source_sha256}\nfeature_schema_sha256={feature_schema_sha256}")
print(f"Nguồn: {csv_path}\nKích thước: {raw.shape[0]:,} dòng × {len(features)} đặc trưng")
print(f"Trùng dòng sau khi bỏ cột chỉ số: {raw.duplicated().sum():,}")
display(raw.head(5))

### 2.1. Tỷ lệ nhãn và giá trị thiếu

In [ ]:
label_table = raw[TARGET].value_counts().sort_index().rename_axis("Nhãn").to_frame("Số dòng")
label_table["Tỷ lệ (%)"] = 100 * label_table["Số dòng"] / len(raw)
missing_table = raw[features].isna().sum().sort_values(ascending=False).to_frame("Số thiếu")
missing_table["Tỷ lệ thiếu (%)"] = 100 * missing_table["Số thiếu"] / len(raw)
display(label_table, missing_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=raw, x=TARGET, ax=axes[0], color="#3977a8")
axes[0].set(title="Phân bố nhãn trên toàn bộ dữ liệu", xlabel="Nhãn (1 = trễ hạn)", ylabel="Số khách hàng")
missing_table["Tỷ lệ thiếu (%)"].sort_values().plot.barh(ax=axes[1], color="#d6873b")
axes[1].set(title="Tỷ lệ thiếu theo đặc trưng", xlabel="Thiếu (%)", ylabel="")
plt.tight_layout()
plt.show()

### 2.2. Phân bố và ngoại lệ

In [ ]:
display(raw[features].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T.round(2))
plot_features = ["age", "RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.flat, plot_features):
    upper = raw[col].quantile(0.99)
    series = raw[col].dropna().clip(upper=upper)
    sns.histplot(series, bins=40, ax=ax, color="#3977a8")
    ax.set(title=f"{col} (hiển thị đến P99)", xlabel=col, ylabel="Số khách hàng")
fig.suptitle("Phân bố đặc trưng; chỉ giới hạn biểu đồ ở P99, không sửa dữ liệu huấn luyện", y=1.02)
plt.tight_layout()
plt.show()

### 2.3. Quan hệ thô với nhãn

In [ ]:
selected = ["age", "RevolvingUtilizationOfUnsecuredLines", "NumberOfTimes90DaysLate",
            "NumberOfTime30-59DaysPastDueNotWorse", "DebtRatio", "MonthlyIncome"]
medians = raw.groupby(TARGET)[selected].median().T
medians.columns = ["Nhãn 0", "Nhãn 1"]
display(medians.round(2))
# Ngoại lệ được giới hạn khi vẽ, các số trong bảng vẫn là trung vị gốc.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col in zip(axes, selected[:3]):
    cap = raw[col].quantile(0.99)
    view = raw[[col, TARGET]].copy()
    view[col] = view[col].clip(upper=cap)
    sns.boxplot(data=view, x=TARGET, y=col, showfliers=False, ax=ax, color="#81b4d0")
    ax.set(xlabel="Nhãn", ylabel=col, title=f"{col} theo nhãn")
plt.tight_layout()
plt.show()

## 3. Chia dữ liệu trước khi huấn luyện

Chia phân tầng 80% train+validation và 20% test. Từ train+validation,
chọn 25% làm validation bằng cùng `split_seed=42`, được 60/20/20.
Grid Search chỉ dùng Train 60%. Mỗi fold tự tính median qua `SimpleImputer` trong pipeline.
Validation chỉ chọn ngưỡng; test giữ nguyên tỷ lệ nhãn và chỉ dùng để đánh giá cuối cùng.
Không dùng SMOTE trên test.

In [ ]:
X = raw[features]
y = raw[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=VALIDATION_FRACTION_OF_TRAIN,
    stratify=y_train, random_state=SEED
)
split_summary = pd.DataFrame({
    "Tập": ["Train", "Validation", "Test"],
    "Số dòng": [len(y_fit), len(y_val), len(y_test)],
    "Tỷ lệ nhãn 1 (%)": [100 * s.mean() for s in [y_fit, y_val, y_test]],
})
display(split_summary.round(2))
assert set(X_fit.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert len(X_fit) + len(X_val) + len(X_test) == len(raw)
print(f"split_seed={SEED}; train/validation/test row IDs tách biệt")
# Trùng nội dung khác với trùng source row ID; giữ nguyên dữ liệu ở baseline chính.
split_labels = pd.Series("test", index=raw.index)
split_labels.loc[X_fit.index] = "train"
split_labels.loc[X_val.index] = "validation"
duplicate_groups = raw.duplicated(subset=features + [TARGET], keep=False)
print("Số dòng thuộc nhóm trùng nội dung theo tập:")
display(split_labels[duplicate_groups].value_counts().rename("Số dòng"))
duplicate_view = raw.loc[duplicate_groups, features + [TARGET]].copy()
duplicate_view["_split"] = split_labels.loc[duplicate_groups]
cross_split_groups = duplicate_view.groupby(features + [TARGET], dropna=False)["_split"].nunique()
print(f"Nhóm nội dung trùng xuất hiện ở nhiều tập: {(cross_split_groups > 1).sum():,}")

## 4. Grid Search Random Forest

Thước đo tìm tham số là **Average Precision (AP)** vì nhãn dương hiếm.
Đây là thí nghiệm tuning riêng; muốn đối chiếu FL, phải khóa cùng bộ tham số
được chọn ở đây cho Centralized và mọi Client, với tổng 100 cây (20 × 5).
Grid Search dùng tối đa 4 tiến trình; mô hình con dùng `n_jobs=1` để tránh lồng nhiều tiến trình.

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        random_state=SEED, max_features="sqrt", n_jobs=1
    )),
])
param_grid = {
    "rf__n_estimators": [100],
    "rf__max_depth": [10, 12],
    "rf__min_samples_leaf": [2, 5],
    "rf__class_weight": [None, "balanced_subsample"],
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
grid = GridSearchCV(
    pipeline, param_grid, scoring="average_precision", cv=cv,
    n_jobs=GRID_N_JOBS, refit=True, verbose=1, return_train_score=False,
    error_score="raise", pre_dispatch="2*n_jobs",
)
grid.fit(X_fit, y_fit)
print("Tham số tốt nhất:", grid.best_params_)
print(f"CV Average Precision: {grid.best_score_:.4f}")

### 4.1. So sánh các cấu hình

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
cols = ["rank_test_score", "mean_test_score", "std_test_score",
        "param_rf__n_estimators", "param_rf__max_depth",
        "param_rf__min_samples_leaf", "param_rf__class_weight"]
display(cv_results[cols].sort_values("rank_test_score").head(10).round(4))

## 5. Chọn ngưỡng trên validation

Báo cáo ba điểm vận hành: ngưỡng mặc định 0,5; ngưỡng tối đa F1;
và ngưỡng ưu tiên precision với recall tối thiểu 30%.
Các ngưỡng được cố định trước khi xem tập test.

In [ ]:
val_scores = grid.best_estimator_.predict_proba(X_val)[:, 1]
p_curve, r_curve, candidate_thresholds = precision_recall_curve(y_val, val_scores)
p_points, r_points = p_curve[:-1], r_curve[:-1]
f1_points = 2 * p_points * r_points / np.maximum(p_points + r_points, 1e-12)
best_f1_index = int(np.argmax(f1_points))
eligible = np.flatnonzero(r_points >= MIN_RECALL_FOR_PRECISION)
if len(eligible) == 0:
    raise ValueError("Không có ngưỡng đạt recall tối thiểu")
precision_index = int(max(eligible, key=lambda i: (p_points[i], r_points[i])))
thresholds = {
    "Mặc định": 0.5,
    "F1 cao nhất": float(candidate_thresholds[best_f1_index]),
    "Ưu tiên precision": float(candidate_thresholds[precision_index]),
}

def point_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"Ngưỡng": threshold, "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0),
            "F1": f1_score(y_true, pred, zero_division=0),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}

validation_metrics = pd.DataFrame({
    name: point_metrics(y_val, val_scores, threshold)
    for name, threshold in thresholds.items()
}).T
print(f"Validation AP = {average_precision_score(y_val, val_scores):.4f}")
display(validation_metrics.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r_curve, p_curve, label=f"RF (AP={average_precision_score(y_val, val_scores):.3f})")
ax.axhline(y_val.mean(), color="gray", linestyle="--", label=f"Tỷ lệ nhãn 1={y_val.mean():.3f}")
for name, threshold in thresholds.items():
    row = validation_metrics.loc[name]
    ax.scatter(row["Recall"], row["Precision"], s=65,
               label=f"{name}: ngưỡng {threshold:.3f}")
ax.set(title="Precision–Recall trên validation", xlabel="Recall của nhãn 1", ylabel="Precision của nhãn 1",
       xlim=(0, 1), ylim=(0, 1))
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Đánh giá mô hình đã fit trên Train 60%

Theo Protocol v1.0, **không refit trên Validation**. Mô hình `grid.best_estimator_`
đã fit trên 90.000 dòng Train; median trong pipeline cũng chỉ học từ Train.
Validation chọn ngưỡng, còn Test 20% chỉ dùng để đánh giá cuối cùng.

In [ ]:
final_model = grid.best_estimator_
assert final_model.named_steps["rf"].n_estimators == 100
test_scores = final_model.predict_proba(X_test)[:, 1]
test_ap = average_precision_score(y_test, test_scores)
test_auc = roc_auc_score(y_test, test_scores)
test_metrics = pd.DataFrame({
    name: point_metrics(y_test, test_scores, threshold)
    for name, threshold in thresholds.items()
}).T
print(f"Test AP = {test_ap:.4f}; ROC-AUC = {test_auc:.4f}")
display(test_metrics.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
PrecisionRecallDisplay.from_predictions(y_test, test_scores, ax=axes[0], name=f"RF (AP={test_ap:.3f})")
axes[0].axhline(y_test.mean(), color="gray", linestyle="--", label="Tỷ lệ nhãn 1")
axes[0].set_title("Precision–Recall trên test gốc")
axes[0].legend()
RocCurveDisplay.from_predictions(y_test, test_scores, ax=axes[1], name=f"RF (AUC={test_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC trên test gốc")
plt.tight_layout()
plt.show()

In [ ]:
chosen_name = "F1 cao nhất"  # Đổi thành "Ưu tiên precision" nếu mục tiêu nghiệp vụ phù hợp.
test_prediction = (test_scores >= thresholds[chosen_name]).astype(int)
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction, labels=[0, 1],
                                        display_labels=["Không trễ hạn", "Trễ hạn"],
                                        cmap="Blues", colorbar=False, ax=ax)
ax.set_title(f"Ma trận nhầm lẫn trên test — {chosen_name}")
plt.tight_layout()
plt.show()

## 7. Kết luận và giới hạn

Ô dưới in kết quả thực tế sau khi chạy. Precision và recall cần được đọc cùng nhau:
tăng ngưỡng thường giảm báo động sai nhưng cũng bỏ sót thêm trường hợp trễ hạn.
Đây là chia ngẫu nhiên cùng bộ dữ liệu, chưa kiểm tra trên khách hàng thuộc giai đoạn khác.
Grid Search ở đây chọn tham số bằng CV trên Train; cùng tham số phải được khóa
cho các Client trước khi so sánh. Split độc lập trong notebook cần đối chiếu
checksum/row IDs với artifact của pipeline chung trước khi gọi đây là cặp đối chứng.
Notebook này chưa huấn luyện Client, ghép cây, đo communication cost hay chạy Azure.
Tập test trong dự án có thể đã từng được dùng ở các thí nghiệm trước; không xem đây là
phép đo hoàn toàn độc lập để công bố hiệu năng triển khai.

In [ ]:
best_row = test_metrics.loc[chosen_name]
print(f"Grid Search chọn: {grid.best_params_}")
print(f"CV AP={grid.best_score_:.4f} | Test AP={test_ap:.4f} | Test ROC-AUC={test_auc:.4f}")
print(f"Điểm vận hành '{chosen_name}' (ngưỡng={thresholds[chosen_name]:.4f}): "
      f"precision={best_row['Precision']:.3f}, recall={best_row['Recall']:.3f}, "
      f"F1={best_row['F1']:.3f}; TP={int(best_row['TP'])}, FP={int(best_row['FP'])}, "
      f"FN={int(best_row['FN'])}.")